# Audit shared-drive disk usage

The microscope computer's data drives are shared across the lab, each laid out as
`{root}/{lab_member}/{sample_dir}`. This notebook scans a list of roots (drive
letters, or specific folders on a drive), measures the on-disk size and creation
date of every `sample_dir`, and sorts the results two ways -- oldest first and
largest first -- so you know who to ask to free up space.

The scan/measure logic lives in `MERci.disk_audit` so it can also be called from
other code.

In [ ]:
import os
import sys
from pathlib import Path

MERCI_DIR = Path(os.getcwd()).parent.parent   # MERci/ (notebook lives in MERci/notebooks/misc/)
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.disk_audit import audit_disk_usage, group_by_lab_member

print(f"MERCI_DIR : {MERCI_DIR}")

## Roots to scan

Each entry is a path expected to directly contain one folder per lab member,
which in turn contains one folder per experiment sample
(`{root}/{lab_member}/{sample_dir}`). An entry can be a whole drive (e.g.
`"D:/"`) or a specific folder on a drive if you only want to audit part of it
(e.g. `"E:/archive/"`). A root that doesn't exist (e.g. a disconnected drive) is
skipped with a warning rather than raising.

In [ ]:
ROOTS = [
    "D:/",
    "E:/",
    "F:/",
]

## Run the scan

Recursively sums file sizes and tracks file-timestamp range for every sample
folder. This walks every file under every root, so it can take a while (minutes)
on drives holding many large experiments -- progress is printed per sample
folder so a long scan doesn't look hung.

In [ ]:
df = audit_disk_usage(ROOTS)
print(f"\nFound {len(df)} sample folder(s).")
df

## Size threshold

Only show sample folders at or above `MIN_SIZE_GB` in the two sorted views
below -- useful once the table gets long and you only care about the folders
actually worth asking someone to delete. Set to `0` to show everything.

In [ ]:
MIN_SIZE_GB = 0   # e.g. 50 to only show sample folders of 50 GB or more

filtered_df = df[df["size_gb"] >= MIN_SIZE_GB] if not df.empty else df
print(f"{len(filtered_df)} of {len(df)} sample folder(s) are >= {MIN_SIZE_GB} GB.")

## Sorted by age -- oldest first

`created` is the sample folder's own creation time on this disk (Windows
`st_ctime`). `earliest_file_modified`/`latest_file_modified` are the oldest/newest
file `mtime` found inside it -- a useful cross-check if the folder itself was
touched (e.g. renamed) after the data was originally copied in.

In [ ]:
cols = ["root", "lab_member", "sample_dir", "size_gb", "created",
        "days_since_created", "n_files"]

if filtered_df.empty:
    print("No sample folders found -- check ROOTS and MIN_SIZE_GB above.")
else:
    display(filtered_df.sort_values("created")[cols].reset_index(drop=True))

## Sorted by size -- largest first

In [ ]:
if filtered_df.empty:
    print("No sample folders found -- check ROOTS and MIN_SIZE_GB above.")
else:
    display(filtered_df.sort_values("size_gb", ascending=False)[cols].reset_index(drop=True))

## Per lab member -- one table each

Same filtered set, split out one table per lab member (largest folder first
within each), with each member's folder count and total size as a header --
so you can copy just their section into an email to them. Grouping is
case-insensitive (`Didar`/`didar` merge into one person) via
`group_by_lab_member`; if more than one casing was found for the same
person, the header lists all of them (e.g. `Didar / didar`) so the merge is
visible.

In [ ]:
if filtered_df.empty:
    print("No sample folders found -- check ROOTS and MIN_SIZE_GB above.")
else:
    for display_name, group in group_by_lab_member(filtered_df):
        print(f"\n{display_name}  --  {len(group)} folder(s), {group['size_gb'].sum():.1f} GB total")
        display(group.sort_values("size_gb", ascending=False)[cols].reset_index(drop=True))

## Optional: save the full table to CSV

Uncomment to write a timestamped snapshot you can share with the lab (e.g. paste
into an email asking people to clear out old data).

In [ ]:
# import datetime
# out_path = MERCI_DIR / f"disk_usage_audit_{datetime.datetime.now():%Y%m%d_%H%M}.csv"
# df.sort_values("created").to_csv(out_path, index=False)          # full scan
# # filtered_df.sort_values("created").to_csv(out_path, index=False)  # only >= MIN_SIZE_GB
# print(f"Saved: {out_path}")